In [1]:
import pandas as pd
from pathlib import Path

In [91]:
ROOT_DIR = Path.cwd()
orders = ROOT_DIR/'data'/'orders_task.csv'
customers = ROOT_DIR/'data'/'customers_task.csv'
order_items = ROOT_DIR/'data'/'order_items_task.csv'
products = ROOT_DIR/'data'/'products_task.csv'

df1 = pd.read_csv(orders)
df2 = pd.read_csv(customers)
df3 = pd.read_csv(order_items)
df4 = pd.read_csv(products)

df5 = pd.merge(df1, df2, how='left', on='customer_id')
df6 = pd.merge(df5, df3, how='left', on='order_id')
df7 = pd.merge(df6, df4, how='left', on='product_id')
df7

df = df7[df7['status'] == 'Completed']
df

,order_id,customer_id,order_date,status,name,country,city,segment,signup_date,item_id,product_id,quantity,product_name,category,price,cost
2,O0002,C022,2024-02-08,Completed,Greta W.,Nepal,Pokhara,Corporate,2023-05-03,L0003,P08,6,Bookshelf,Furniture,313.58,175.71
3,O0002,C022,2024-02-08,Completed,Greta W.,Nepal,Pokhara,Corporate,2023-05-03,L0004,P05,10,Power Bank,Electronics,297.41,172.35
4,O0003,C013,2024-09-15,Completed,Sofia N.,Australia,Melbourne,Home Office,2023-04-23,L0005,P09,3,Filing Cabinet,Furniture,179.75,143.54
5,O0004,C028,2024-03-09,Completed,Zoe C.,Germany,Munich,Consumer,2023-01-28,L0006,P18,5,Lunch Box,Kitchen,103.66,58.49
6,O0004,C028,2024-03-09,Completed,Zoe C.,Germany,Munich,Consumer,2023-01-28,L0007,P06,2,Office Chair,Furniture,359.18,225.18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
902,O0299,C002,2024-01-26,Completed,Rajesh C.,India,Delhi,Consumer,2023-12-04,L0903,P04,7,Webcam,Electronics,107.78,69.26
903,O0299,C002,2024-01-26,Completed,Rajesh C.,India,Delhi,Consumer,2023-12-04,L0904,P23,3,Jump Rope,Sports,94.74,59.41
904,O0299,C002,2024-01-26,Completed,Rajesh C.,India,Delhi,Consumer,2023-12-04,L0905,P16,2,Coffee Mug,Kitchen,136.76,103.65
905,O0300,C016,2024-03-26,Completed,Ravi Q.,USA,Austin,Home Office,2023-06-06,L0906,P13,2,Sticky Notes,Stationery,18.47,10.51


In [59]:
df.columns = df.columns.str.strip()

1. Report total realized revenue, total profit, and the overall profit margin.

In [92]:
df['revenue'] = df['quantity'] * df['price']
df['profit'] = df['revenue'] - (df['cost']*df['quantity'])
df['profit_margin'] = df['profit'] / df['revenue']
rev = df['revenue'].sum()
prof = df['profit'].sum()
profit_margin = prof/rev*100
print(f"Revenue: {rev:.2f}\nProfit: {prof:.2f}\nProfit Margin: {profit_margin:.2f}%")

Revenue: 666213.57
Profit: 246609.45
Profit Margin: 37.02%


In [34]:
sum_df = pd.DataFrame({
    'missing': df.isnull().sum(),
    'unique': df.nunique(),
    'dtype': df.dtypes
})
sum_df

,missing,unique,dtype
order_id,0,250,str
customer_id,0,35,str
order_date,0,176,str
status,0,1,str
name,0,35,str
country,0,6,str
city,0,15,str
segment,0,3,str
signup_date,0,32,str
item_id,0,740,str


In [86]:
df = df.reset_index()

2. Which product category earns the most revenue? Rank all categories. 

In [53]:
df = df.groupby('category')['revenue'].sum().sort_values(ascending=False)
df

category
Electronics    278762.65
Furniture      232074.07
Sports          73040.38
Kitchen         70333.29
Stationery      12003.18
Name: revenue, dtype: float64

In [87]:
print(df.columns.to_list())

['product_name', 'revenue']


3. What are the top 10 products by revenue?

In [94]:
t10 = df.groupby('product_name')['revenue'].sum().sort_values(ascending=False)
t10 = t10.head(10)
t10

product_name
Bluetooth Speaker    92183.45
Desk Lamp            69974.28
Office Chair         66807.48
Wireless Mouse       63801.42
Power Bank           59482.00
Bookshelf            52681.44
USB-C Hub            47559.90
Yoga Mat             28346.55
Lunch Box            26744.28
Filing Cabinet       23547.25
Name: revenue, dtype: float64

4. Compute revenue and average order value by customer segment.

In [69]:
segment = df.groupby('segment').agg(
    total=('revenue', 'sum'),
    avg=('revenue', 'mean'),
)
segment

,total,avg
segment,,
Consumer,217227.20,838.715058
Corporate,57530.97,991.913276
Home Office,391455.40,925.426478


5. Compute revenue and average order value by country.

In [70]:
country = df.groupby('country').agg(
    total=('revenue', 'sum'),
    avg=('revenue', 'mean'),
)
country

,total,avg
country,,
Australia,157176.20,863.605495
Germany,119305.18,817.158767
India,53320.48,903.736949
Nepal,126236.89,1026.316179
UK,23309.74,971.239167
USA,186865.08,907.112039


6. What is the total revenue for each month?

In [71]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['month'] = df['order_date'].dt.month_name()
mdf = df.groupby('month')['revenue'].sum()
mdf

month
April        43080.06
August       57580.97
December     56822.45
February     54394.34
January      50473.08
July         47928.45
June         50156.36
March        45684.58
May          61294.54
November     72757.25
October      84981.72
September    41059.77
Name: revenue, dtype: float64

7. Who are the top 5 customers by total spend?

In [72]:
tt5 = df.groupby('customer_id').agg(total=('revenue', 'sum'))
t5 = tt5.sort_values('total', ascending=False).head()
t5

,total
customer_id,
C010,35273.93
C020,34015.30
C032,33723.96
C029,33586.08
C013,33252.26


8. Which products were sold in the greatest number of orders?

In [73]:
mprod = df.groupby('product_name')['quantity'].sum()
mprod.sort_values(ascending=False)

product_name
Lunch Box            258
Desk Lamp            243
Bluetooth Speaker    235
Water Bottle         232
Pen Set              212
Power Bank           200
Jump Rope            198
Office Chair         186
Marker Pack          185
Water Flask          178
Yoga Mat             177
Notebook             177
USB-C Hub            177
Bookshelf            168
Dumbbell Set         157
Sticky Notes         155
Kettle               155
Coffee Mug           154
Webcam               146
Cutting Board        139
Wireless Mouse       134
Filing Cabinet       131
Standing Desk        129
Stapler              101
Name: quantity, dtype: int64

9. What is the average number of items per order?

In [74]:
qitem = df.groupby('order_id').size().mean()
print(qitem)

2.96


10. What percentage of all orders were Cancelled or Returned?

In [75]:
percent_un = (df7['status']!='Completed').mean()*100
print(percent_un)

18.412348401323044


11. Which customers have never placed an order, and which products were
never sold?

In [76]:
never_order_customer = df2[~df2['customer_id'].isin(df1['customer_id'])]
print(never_order_customer)

   customer_id      name  country       city      segment signup_date
35        C036  Kiran K.    Nepal   Lalitpur    Corporate  2023-11-17
36        C037   Anil L.  Germany     Berlin     Consumer  2023-06-25
37        C038   Rita M.    India  Bangalore  Home Office  2023-01-20
38        C039  Suraj N.      USA     Austin     Consumer  2023-02-12
39        C040   Puja O.      USA   New York     Consumer  2023-04-19


In [77]:
never_order_product = df4[~df4['product_id'].isin(df3['product_id'])]
print(never_order_product)

   product_id      product_name category   price    cost
24        P25  Resistance Bands   Sports  139.57  110.99


In [78]:
type(df)

pandas.DataFrame

12. For each country, which product sold the most by revenue? 

In [79]:
result = df.loc[df.groupby('country')['revenue'].idxmax()]
print(result[['product_name', 'country']])


          product_name    country
60      Wireless Mouse  Australia
161  Bluetooth Speaker    Germany
687  Bluetooth Speaker      India
513  Bluetooth Speaker      Nepal
686       Office Chair         UK
461     Wireless Mouse        USA


13. Which single customer generated the most revenue, and from which
country?

In [104]:
most_customer = df.groupby(['customer_id', 'name', 'country'], as_index=False)['revenue'].sum()
most_customer = most_customer.sort_values(ascending=False, by='revenue').head(1)
most_customer

,customer_id,name,country,revenue
9,C010,Olivia K.,USA,35273.93


14. Which product category is the most profitable?

In [106]:
pro_cat = df.groupby('category')['revenue'].sum().sort_values(ascending=False)
pro_cat.head(1)

category
Electronics    278762.65
Name: revenue, dtype: float64

15. Which product has the highest profit margin (%)?

In [110]:
high_pro = df.groupby('product_name')['profit_margin'].mean().sort_values(ascending=False)
high_pro.head(1)

product_name
USB-C Hub    0.479233
Name: profit_margin, dtype: float64